In [9]:
import os
import librosa
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

DATASET_PATH = r"C:\Users\ADMIN\Desktop\Sukriti\VIT\SL\AnuAssistant\dataset"

features = []
labels = []

print("Loading dataset...")

for label in os.listdir(DATASET_PATH):

    folder = os.path.join(DATASET_PATH, label)

    if not os.path.isdir(folder):
        continue

    for file in os.listdir(folder):

        if file.endswith(".wav"):

            file_path = os.path.join(folder, file)

            try:
                # Load audio
                audio, sr = librosa.load(file_path, sr=16000)

                # normalize volume
                audio = librosa.util.normalize(audio)

                # remove silence
                audio, _ = librosa.effects.trim(audio, top_db=20)

                # force fixed duration (1 second)
                TARGET_SAMPLES = 16000

                if len(audio) < TARGET_SAMPLES:
                    audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
                else:
                    audio = audio[:TARGET_SAMPLES]



                # Extract MFCC (40 coefficients)
                mfcc = librosa.feature.mfcc(
                    y=audio,
                    sr=sr,
                    n_mfcc=40
                )

                # --- FIX LENGTH (IMPORTANT) ---
                # Ensure all samples same size (40 x 32)

                TARGET_LENGTH = 64   # longer audio window

                if mfcc.shape[1] < TARGET_LENGTH:
                    pad_width = TARGET_LENGTH - mfcc.shape[1]
                    mfcc = np.pad(
                        mfcc,
                        ((0, 0), (0, pad_width)),
                        mode='constant'
                    )
                else:
                    mfcc = mfcc[:, :TARGET_LENGTH]


                features.append(mfcc)
                labels.append(label)

            except Exception as e:
                print("Error processing:", file_path, e)

print("Feature extraction complete.")

# Convert to numpy arrays
X = np.array(features)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Create models folder if not exists
os.makedirs("models", exist_ok=True)

# Save encoder
with open("models/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Save dataset
np.save("models/X.npy", X)
np.save("models/y.npy", y_encoded)

print("✅ Data saved successfully in models folder.")


Loading dataset...
Feature extraction complete.
X shape: (740, 40, 64)
y shape: (740,)
✅ Data saved successfully in models folder.
